In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
import seaborn as sns
from sklearn.metrics import accuracy_score, recall_score,precision_score,f1_score,confusion_matrix
from matplotlib import pyplot as plt

In [ ]:
df=pd.read_csv("penguins_train.csv")
df_generate=pd.read_csv("penguins_test.csv")
df

In [ ]:
df.info()

In [ ]:
y=df['Species']
df=df.drop('Species',axis=1)

In [ ]:
imputer=SimpleImputer(strategy='mean')
cols= ['Culmen Length (mm)','Culmen Depth (mm)','Flipper Length (mm)','Body Mass (g)','Delta 15 N (o/oo)','Delta 13 C (o/oo)']
df[cols]=imputer.fit_transform(df[cols])
df_generate[cols]=imputer.fit_transform(df_generate[cols])

imputer= SimpleImputer(strategy='most_frequent')
cols=['Sex']
df[cols]=imputer.fit_transform(df[cols])
df_generate[cols]=imputer.fit_transform(df_generate[cols])

In [ ]:
df=pd.get_dummies(df,drop_first=True)
df_generate=pd.get_dummies(df_generate,drop_first=True)
df.drop(columns=["Sex_FEMALE"],inplace=True,axis=0)

In [ ]:
df

In [ ]:
scaler=StandardScaler()
columns=df.columns[0:6]
df[columns]=scaler.fit_transform(df[columns])
df_generate[columns]=scaler.fit_transform(df_generate[columns])

print(df.columns,df_generate.columns)

In [ ]:
class RandomForestClassifier():
    def __init__(self,n_iter):
        self.n_iter=n_iter
        self.trees=[]
    
    def bootstrap(self,X,y):
        n_samples=X.shape[0]
        idx=np.random.choice(n_samples,n_samples,replace=True)
        return X[idx],y[idx]
        
    def fit(self,X_train,y_train):
        for _ in range(self.n_iter):
            X,y=self.bootstrap(X_train,y_train)
            tree= DecisionTreeClassifier(criterion="entropy",splitter="best",max_features="sqrt",random_state=42)
            tree.fit(X,y)
            self.trees.append(tree)
    
    def predict(self,X):
        tree_preds = np.array([tree.predict(X) for tree in self.trees])
        pred=[]
        for col_idx in range(tree_preds.shape[1]):
            unique_values, counts = np.unique(tree_preds[:,col_idx], return_counts=True)
            mode_index = np.argmax(counts)
            pred.append(unique_values[mode_index])
        return pred

In [ ]:
X_train,X_test,y_train,y_test= train_test_split(df,y,test_size=0.2,random_state=42)

In [ ]:
n_values= np.arange(5,41,5)
accuracy=[]
for n in n_values:
    classifier= RandomForestClassifier(10)
    classifier.fit(X_train.values,y_train.values)
    pred=classifier.predict(X_test.values)
    accuracy.append(accuracy_score(y_test,pred))

plt.plot(n_values,accuracy)
plt.xlabel("N Values")
plt.ylabel("Accuracy")
plt.title("Accuracy vs Number of decision trees")
plt.show()

### `Best Number of Decision Trees`

In [ ]:
n_max=n_values[np.argmax(accuracy)]
print(n_max)

In [ ]:
classifier= RandomForestClassifier(n_max)
classifier.fit(X_train.values,y_train.values)
pred=classifier.predict(X_test.values)

In [ ]:
pred

In [ ]:
print("Accuracy : ",accuracy_score(y_test,pred))
print("Macro Precision : ",precision_score(y_test,pred,average="macro"))
print("Micro Precision : ",precision_score(y_test,pred,average="micro"))
print("Macro Recall : ",recall_score(y_test,pred,average="macro"))
print("Micro Recall : ",recall_score(y_test,pred,average="micro"))
print("F1-Score Macro : ",f1_score(y_test,pred,average="macro"))
print("F1-Score Micro : ",f1_score(y_test,pred,average="micro"))

In [ ]:
cm = confusion_matrix(y_test, pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", linewidths=1, linecolor="k", cbar=False,)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
pred=classifier.predict(df_generate.values)

In [ ]:
with open('random.csv', 'w') as f:
    f.write("predicted\n")
    for item in pred:
        f.write("%s\n" % item)